
# A Minimal LangGraph ReAct Agent

**Day 3 — RAG & Agents · Practical 5 of 6 · Companion to the "Agent Fundamentals" deck**

> **Running in Google Colab:** works on the default **CPU runtime** — this notebook calls an
> LLM API, no local model inference or GPU needed.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Build a LangGraph graph with an Agent node, a Tools node, and conditional routing
2. Give the agent a real tool it can call, and watch it decide when to call it
3. See a full ReAct-style trace: Thought -> Action -> Observation -> ... -> Final Answer

## Why This Matters for a Law Firm

This is the exact minimal skeleton — Agent node, Tools node, one conditional edge — underneath
nearly every production tool-using agent, including the multi-agent system in the next
notebook. Understanding this small graph is the prerequisite for everything more complex.

## Notebook Workflow

```mermaid
flowchart TD
    Start(["Start"]) --> Agent["Agent node\n(LLM decides)"]
    Agent -->|"tool call requested"| Tools["Tools node\n(executes tool)"]
    Agent -->|"no tool call / done"| End(["End"])
    Tools --> Agent



## Section 1 — Setup


In [ ]:

%pip install -q langgraph langchain-openai

import os

def get_api_key(env_var_name):
    try:
        from google.colab import userdata
        key = userdata.get(env_var_name)
        if key:
            return key
    except ImportError:
        pass
    return os.environ.get(env_var_name)

OPENAI_API_KEY = get_api_key("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Add an OPENAI_API_KEY secret in Colab (key icon, left sidebar).")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API key configured.")



## Section 2 — Define a Legal Tool

A mock "clause lookup" tool: given a clause topic, it returns the matching section from a small
hardcoded contract. In a real system this would query the RAG pipeline from earlier notebooks
instead of a hardcoded dictionary -- the agent-side code is identical either way.


In [ ]:

from langchain_core.tools import tool

CLAUSE_DATABASE = {
    "indemnification": "Section 7.1: The Contractor shall indemnify the Client for claims arising from gross negligence.",
    "termination": "Section 9.2: Either party may terminate this Agreement upon sixty (60) days' written notice.",
    "liability": "Section 14.3: Total liability shall not exceed fees paid in the preceding twelve (12) months.",
    "governing law": "Section 16.1: This Agreement is governed by the laws of the State of Delaware.",
}

@tool
def lookup_clause(topic: str) -> str:
    '''Look up a contract clause by topic (e.g. 'indemnification', 'termination', 'liability', 'governing law').'''
    topic_key = topic.lower().strip()
    for key, clause_text in CLAUSE_DATABASE.items():
        if key in topic_key or topic_key in key:
            return clause_text
    return f"No clause found for topic: {topic!r}. Available topics: {list(CLAUSE_DATABASE.keys())}"

# Quick manual test of the tool itself, outside the agent
print(lookup_clause.invoke({"topic": "termination"}))



## Section 3 — Build the Graph

Exactly the deck's "Putting It Together" structure: an Agent node, a Tools node, and a
conditional edge routing between them based on whether the model requested a tool call.


In [ ]:

from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

tools = [lookup_clause]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

def agent_node(state: AgentState):
    # Call the LLM with the conversation so far; it may respond with a tool call or a final answer
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    # If the model's last response requested a tool call, route to the Tools node; else, end
    return "tools" if last_message.tool_calls else END

graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")  # after a tool runs, go back to the agent to decide next steps

app = graph.compile()
print("Graph compiled: agent <-> tools loop with conditional routing to END.")



## Section 4 — Run the Agent

Ask a question that requires the tool. Watch the full message trace, including the tool call
the model requested and the tool's result.


In [ ]:

from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [HumanMessage(content="What does our contract say about termination?")]
})

for msg in result["messages"]:
    role = msg.__class__.__name__
    content = msg.content if msg.content else f"[tool call: {msg.tool_calls}]" if hasattr(msg, "tool_calls") and msg.tool_calls else "[no content]"
    print(f"{role}: {content}\n")



## Section 5 — A ReAct-Style Trace, Made Explicit

The deck's ReAct pattern interleaves explicit Thought/Action/Observation steps. LangGraph's
tool-calling loop implements this same cycle, just with the "Thought" implicit in the model's
tool-call decision rather than a separate labeled step. Let's make it explicit by printing the
trace in ReAct format.


In [ ]:

def run_and_print_react_trace(question):
    result = app.invoke({"messages": [HumanMessage(content=question)]})

    print(f"Question: {question}\n")
    for msg in result["messages"][1:]:  # skip the initial human question
        cls_name = msg.__class__.__name__
        if cls_name == "AIMessage" and getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                print(f"Thought: I should look up the '{tc['args'].get('topic', '?')}' clause.")
                print(f"Action: {tc['name']}({tc['args']})")
        elif cls_name == "ToolMessage":
            print(f"Observation: {msg.content}")
        elif cls_name == "AIMessage" and msg.content:
            print(f"Final Answer: {msg.content}")
    print("\n" + "-" * 70 + "\n")

run_and_print_react_trace("What is our liability cap under the agreement?")



## Section 6 — Try It Yourself

Ask about a topic NOT in the clause database, and watch the agent handle the tool's "not found"
response gracefully rather than making something up.


In [ ]:

run_and_print_react_trace("What does the contract say about late payment penalties?")



## Key Takeaways

1. **The graph is genuinely small**: one Agent node, one Tools node, one conditional edge, one
   loop-back edge -- this is the deck's claim that this minimal skeleton underlies most
   production tool-using agents, made literal.
2. **Routing is state-driven, not hardcoded** -- `should_continue` inspects the LAST message's
   `tool_calls` attribute at runtime; the graph doesn't know in advance whether a query needs
   zero, one, or several tool calls.
3. **The tool itself doesn't need to change** whether it's backed by a hardcoded dictionary (as
   here) or a full RAG pipeline (as in earlier notebooks) -- from the agent's perspective, it's
   just a function that takes arguments and returns text.

**Next up:** the *Multi-Agent Supervisor Pattern* notebook — composing several agents like this
one together.
